# bg

> The bgterm API: sync, cursor-paged background terminal sessions

The [bgterm](https://github.com/AnswerDotAI/bgterm) package, folded in: a *sync*, sid-based API for background terminal sessions — start once, send input later, wait a bounded time, read what arrived since your last look. It is the cursor view of a `PtySession`, for callers that live outside an event loop (kernel tools, plain scripts, an LLM deciding between actions): every session shares one private asyncio loop on a daemon thread, and each call marshals onto it and blocks, so `write_stdin(sid, "2+2\n", 500)` means what it always meant. The buffering, paging, and drop accounting are the `Ring`'s, which was extracted from bgterm in the first place. The wait parameters are `fastmux.bg`'s. `wait_ms` bounds the wait for new output. `until=` returns as soon as the accumulated text matches that regex. `settle_ms` keeps collecting until output has stopped for that long.

In [ ]:
from ptymini.bg import *
from nbdev.showdoc import show_doc


In [ ]:
import sys, time
from fastcore.test import test_eq

## PollResult

Every read-shaped call blocks the calling thread directly against the Rust core (the GIL is released while waiting) and returns the unread output as a `PollResult`. (bgterm previously ran two threads *per session*, and an earlier ptymini one loop thread per process; this runs none.)


In [ ]:
show_doc(PollResult)


## Sessions

`_BgSession` is the cursor over one pty core: it remembers the last-read offset, decodes at the edge, and builds each `PollResult` from the ring's absolute offsets — `dropped_bytes` is the distance the ring's retained window moved past the cursor. String commands run via `/bin/sh -c` (bgterm's `shell=True`), `TERM=dumb` overlays the environment as before, and a signal death reports as a negative exit code, `subprocess` style.

The sid table and functional API — the sid is how a kernel tool names a session across calls. The wait parameters follow `fastmux.bg`, with two differences the substrate forces. Waits are event-driven: a condvar wakes on new bytes or death. There is no `interval_ms`. Reading consumes the stream, and `until` can only match output accumulated within the current call. Match on text only the awaited output can produce, such as a fresh prompt. The whole call is bounded by `wait_ms + settle_ms`.

In [ ]:
show_doc(start_bgterm)


In [ ]:
show_doc(write_stdin)

In [ ]:
show_doc(poll)

In [ ]:
show_doc(read)

In [ ]:
show_doc(wait)

In [ ]:
show_doc(terminate)

In [ ]:
show_doc(kill)

In [ ]:
show_doc(close_bgterm)

In [ ]:
show_doc(list_sessions)

`Session` wraps a sid for `with`-block use, verbatim from bgterm:

In [ ]:
show_doc(Session)


The whole flow, as a reader would use it — start a REPL-ish child, wait for its banner, then send input and name the reply to wait for. `until` replaces the write-then-poll-again loop with one call:

In [ ]:
sid = start_bgterm([sys.executable, '-u', '-c',
    "print('ready', flush=True)\nimport sys\nfor line in sys.stdin: print(f'ACK:{line.strip()}', flush=True)"])
first = poll(sid, 5000)
assert 'ready' in first.text
r = write_stdin(sid, 'hello\n', 2000, until=r'ACK:hello')
assert 'ACK:hello' in r.text
assert sid in list_sessions()
r.text

`settle_ms` is for replies whose shape you don't know. After the wait, the call keeps collecting until the stream has been quiet for that long. A multi-burst reply comes back in one call. A child that dies ends the settle at once.

In [ ]:
sid2 = start_bgterm([sys.executable, '-u', '-c',
    "import time\nprint('part one', flush=True)\ntime.sleep(0.2)\nprint('part two', flush=True)"])
r = poll(sid2, 5000, settle_ms=500)
assert 'part one' in r.text and 'part two' in r.text
close_bgterm(sid2)
r.text

Paging and drop accounting come straight from the ring: a bounded read leaves `remaining_bytes`, a small buffer reports what overflow cost, and `truncated` summarizes both.

In [ ]:
sid2 = start_bgterm([sys.executable, '-u', '-c', "import sys; sys.stdout.write('x'*4096); sys.stdout.flush()"], max_buffer_bytes=512)
time.sleep(0.3)
r = read(sid2, 256)
test_eq((r.bytes_returned, r.dropped_bytes, r.truncated), (256, 4096 - 512, True))
r2 = read(sid2, None)
test_eq((r2.bytes_returned, r2.remaining_bytes), (256, 0))
close_bgterm(sid2)
close_bgterm(sid)
with Session.start([sys.executable, '-c', "print('bye'); raise SystemExit(3)"]) as sess:
    assert 'bye' in sess.poll(3000).text
    test_eq(sess.wait(3000), 3)
list_sessions()